In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
pm_df = con.execute("SELECT * FROM opspatial WHERE total_reception_events >= 15 AND total_release_events >= 15 AND position != 'Goalkeeper'").df()
con.close()

# K-Means Clustering on Positional Data
Seeing if K-Means is a good fit for this data, by using ground truth data (position, player, team).

In [ ]:
pm_df["avg_reception_simpnorm_y"] = np.where(
    pm_df["average_reception_y"] <= 40,
    pm_df["average_reception_y"],
    80 - pm_df["average_reception_y"],
)

pm_df["avg_release_simpnorm_y"] = np.where(
    pm_df["average_release_y"] <= 40,
    pm_df["average_release_y"],
    80 - pm_df["average_release_y"],
)

pm_df["avg_reception_simpnorm_y_abs"] = pm_df["avg_reception_simpnorm_y"].abs()
pm_df["avg_release_simpnorm_y_abs"] = pm_df["avg_release_simpnorm_y"].abs()

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]

In [ ]:
positions = pm_df["position"].unique().copy()
print(positions)

In [ ]:
position_data = pm_df.dropna(
    subset=[
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
        "position",
    ]
).copy()
positions = sorted(position_data["position"].unique())

ncols = 3
nrows = (len(positions) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * nrows),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
position_colors = plt.get_cmap("tab20", len(positions))

for index, position in enumerate(positions):
    ax = axes.flat[index]
    pitch.draw(ax=ax)

    rows = position_data[position_data["position"] == position]
    color = position_colors(index)

    pitch.lines(
        rows["average_reception_x"],
        rows["average_reception_y"],
        rows["average_release_x"],
        rows["average_release_y"],
        color=color,
        alpha=0.3,
        lw=0.8,
        ax=ax,
    )
    pitch.scatter(
        rows["average_reception_x"],
        rows["average_reception_y"],
        color=color,
        marker="o",
        s=24,
        ax=ax,
    )
    pitch.scatter(
        rows["average_release_x"],
        rows["average_release_y"],
        color=color,
        marker="^",
        s=28,
        ax=ax,
    )
    ax.set_title(str(position))

for ax in axes.flat[len(positions):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
        Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
    ],
    loc="upper center",
    ncol=2,
)
fig.suptitle("Average reception-to-release locations by position", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
def kmeans_eval(df : pd.DataFrame,
                start : int = 2,
                end : int = 10,
                random_state : int = 42,
                sil_true : bool = True,
                **kmeanskwargs):
    main_df = df.copy()
    inertia = []
    silhouette = []
    for i in range(start, end + 1):
        kmeans = KMeans(n_clusters=i, random_state=random_state, **kmeanskwargs)
        clusters = kmeans.fit_predict(main_df)

        inertia.append(kmeans.inertia_)
        if sil_true:
            score = silhouette_score(main_df,
                                     clusters,
                                     sample_size=min(5000, len(main_df)),
                                     random_state=random_state)
            silhouette.append(score)

    return inertia, silhouette


In [ ]:
pm_cluster = pm_df[feature_cols].copy()
START = 2
END = 10

# Testing Standard Scaler
scaler = StandardScaler()
pm_cluster_standardscaled = scaler.fit_transform(pm_cluster)

pm_inertia, pm_silhouette = kmeans_eval(pm_cluster_standardscaled, START, END)

fig_pm, (ax1_pm, ax2_pm) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm.plot(list(range(START, END + 1)), pm_inertia, marker="o")
ax1_pm.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm.plot(list(range(START, END + 1)), pm_silhouette, marker="o")
ax2_pm.set(title="Silhouette Scores", xlabel="Number of Clusters (k)", ylabel="Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
def standard_kmeans_apply(df : pd.DataFrame,
                 feature_cols : list,
                 column_name : str = "clusters",
                 n_clusters : int = 3,
                 random_state : int = 42,
                 **kmeanskwargs):
    """Add K-Means cluster labels to dataframe."""
    main_df = df.copy()
    target_df = main_df[feature_cols].copy()
    scaler = StandardScaler()
    target_df_scaled = scaler.fit_transform(target_df)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, **kmeanskwargs)
    clusters = kmeans.fit_predict(target_df_scaled)
    main_df[column_name] = clusters
    return main_df

In [ ]:
K_VALS = [3, 4, 5, 6]
FIG_ROW = 2
FIG_COL = 2

fig, axes =  plt.subplots(nrows= FIG_ROW, ncols=FIG_COL, figsize = (12, 9))
pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333", axis=True, label=True, tick=True)
k_clustered_dfs = []
for i, ax in enumerate(axes.flat):
    k = K_VALS[i]
    pitch.draw(ax=ax)
    clusters = standard_kmeans_apply(df=pm_df, feature_cols=feature_cols, column_name=f"clusters_{k}", n_clusters = k)
    colours = plt.get_cmap("tab10", k)
    k_clustered_dfs.append(clusters)

    for n in range(k):
        cluster_rows = clusters[clusters[f"clusters_{k}"] == n]
        colour = colours(n)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=colour,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=colour,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=colour,
            marker="^",
            s=28,
            ax=ax,
        )
    
    cluster_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=colours(n), label=f"Cluster {n}")
        for n in range(k)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial clusters (k={k})")
plt.tight_layout()
plt.show()



Clustering does seem to match the positional "zones" on the pitch. All players are tagged with their position for that match, and we expect that most players will generally stick to a specific role/roles with some outliers (substitutes/camavinga - or fellaini if talking about 15/16)

In [ ]:
def crosstab_cluster(clustered_df : pd.DataFrame, cluster_col : str, target_col : str, **kwargs):
    df = clustered_df.copy()
    main_df = pd.crosstab(df[target_col], df[cluster_col], **kwargs)
    return main_df


In [ ]:
cluster_poscounts = []
for i in range(len(K_VALS)):
    k = K_VALS[i]
    cols = [f"col_{j}" for j in range(k)]
    cur_df = crosstab_cluster(k_clustered_dfs[i].copy(), cluster_col = f"clusters_{k}", target_col = "position")
    cur_df.columns = cols
    cur_df["total_sum"] = cur_df[cols].sum(axis = 1)
    perc_col = [f"percentage_{j}" for j in range(k)]
    cur_df[perc_col] = cur_df[cols].div(cur_df["total_sum"], axis = 0)*100
    cluster_poscounts.append(cur_df)

In [ ]:
cluster_playcounts = []
for i in range(len(K_VALS)):
    k = K_VALS[i]
    cols = [f"col_{j}" for j in range(k)]
    cur_df = crosstab_cluster(k_clustered_dfs[i].copy(), cluster_col = f"clusters_{k}", target_col = "player")
    cur_df.columns = cols
    cur_df["total_sum"] = cur_df[cols].sum(axis = 1)
    perc_col = [f"percentage_{j}" for j in range(k)]
    cur_df[perc_col] = cur_df[cols].div(cur_df["total_sum"], axis = 0)*100
    cluster_playcounts.append(cur_df)

In [ ]:
def column_barplot(target_df : pd.DataFrame,
                   column_name : str,
                   title : str = 'Bar Graph of Column Values',
                   xlabel : str = 'Values',
                   ylabel : str = 'Index'):
    df = target_df.copy()
    df = df.sort_values(by = column_name)
    df[column_name].plot.barh()
    plt.title(title)
    plt.xlim(0,100)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [ ]:
for k in range(3,7):
    for i in range(k):
        column_barplot(
            cluster_poscounts[k-3],
            f"percentage_{i}",
            title=f"Position Percentages for Cluster {i}, k={k}",
            xlabel = "percentages",
            ylabel = "positions"
    )


In [ ]:
def column_scatterplot(target_df : pd.DataFrame,
                   column_name : str,
                   title : str = 'Scatter plot',
                   xlabel : str = 'Index',
                   ylabel : str = 'Values'):
    df = target_df.copy()
    df = df.sort_values(by = column_name)
    plt.scatter(x = range(len(df)), y = df[column_name], s=1, alpha=0.2)
    plt.title(title)
    plt.ylim(0,100)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [ ]:
for k in range(3,7):
    for i in range(k):
        column_scatterplot(
            cluster_playcounts[k-3],
            f"percentage_{i}",
            title=f"Player Percentages for Cluster {i}, k={k}",
            xlabel = "points",
            ylabel = "percentages"
    )

In [ ]:

def column_histplot(target_df : pd.DataFrame, cluster : int = 0, k : int = 0):
    df = target_df.copy()
    
    # 1. CREATE THE FIGURE AND AXES OBJECTS HERE
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Define clean 10% bin edges: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    bin_edges = list(range(0, 101, 10))

    # Plot the histogram (now safely attached to the local ax)
    n, bins, patches = ax.hist(
        df[f"percentage_{cluster}"], 
        bins=bin_edges, 
        color='#3498db',      # Clean blue color
        edgecolor='#2c3e50',  # Dark border for distinct bars
        linewidth=1.2,
        rwidth=0.95           # Adds a tiny gap between bars for better readability
    )

    # Customise titles and labels
    ax.set_title(f'Distribution of Players for Cluster {cluster} k={k}', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(f'Percentage of Matches in Cluster {cluster}', fontsize=12, labelpad=10)
    ax.set_ylabel('Number of Players', fontsize=12, labelpad=10)

    # Align the ticks precisely with your 10% bins
    ax.set_xticks(bin_edges)

    # Clean up layout and add light horizontal grids for tracking counts
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)  # Pushes grid lines behind the bars

    # Display the chart
    plt.tight_layout()
    plt.show()

In [ ]:
for k in range(3,7):
    for i in range(k):
        column_histplot(cluster_playcounts[k-3], i, k)

In [ ]:
def entropy_calc(target_df : pd.DataFrame, columns : list, col_name : str = "entropy", normalise : bool = False, norm_n : int = 2):
    df = target_df.copy()
    p = df[columns].to_numpy()
    log_p = np.where(p > 0, np.log(p), 0)
    entropy = -np.sum(p * log_p, axis=1)
    if normalise:
        np.divide(entropy, norm_n)
    df[col_name] = entropy
    return df

In [ ]:
entropy_dfs = []
for k in range(3,7):
    cur_df = cluster_playcounts[k-3].copy()
    cur_df = cur_df[cur_df["total_sum"] >= 20]
    targets = [f"percentage_{n}" for n in range(k)]
    cur_df[targets] = cur_df[targets].div(100)
    entropy_dfs.append(entropy_calc(cur_df, targets, normalise = True, norm_n = k))

In [ ]:
def column_histplot_2(target_df : pd.DataFrame, k : int = 0):
    df = target_df.copy()
    
    # 1. CREATE THE FIGURE AND AXES OBJECTS HERE
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Define clean 10% bin edges: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    bin_edges = np.arange(0, 1.01, 0.05).tolist()

    # Plot the histogram (now safely attached to the local ax)
    n, bins, patches = ax.hist(
        df[f"entropy"], 
        bins=bin_edges, 
        color='#3498db',      # Clean blue color
        edgecolor='#2c3e50',  # Dark border for distinct bars
        linewidth=1.2,
        rwidth=0.95           # Adds a tiny gap between bars for better readability
    )

    # Customise titles and labels
    ax.set_title(f'Entropy of Players for k={k}', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(f'Bins of Normalised Entropy', fontsize=12, labelpad=10)
    ax.set_ylabel('Number of Players', fontsize=12, labelpad=10)

    # Align the ticks precisely with your 10% bins
    ax.set_xticks(bin_edges)

    # Clean up layout and add light horizontal grids for tracking counts
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)  # Pushes grid lines behind the bars

    # Display the chart
    plt.tight_layout()
    plt.show()

In [ ]:
for k in range(3,7):
    column_histplot_2(entropy_dfs[k-3], k)

In [ ]:
entropy_dfs = []
for k in range(3,7):
    cur_df = cluster_playcounts[k-3].copy()
    cur_df = cur_df[cur_df["total_sum"] >= 20]
    cur_df = cur_df[cur_df["col_2"] >= 1]
    targets = [f"percentage_{n}" for n in range(k)]
    cur_df[targets] = cur_df[targets].div(100)
    entropy_dfs.append(entropy_calc(cur_df, targets, normalise = True, norm_n = k))

In [ ]:
for k in range(3,7):
    column_histplot_2(entropy_dfs[k-3], k)

things to also consider, heatmaps and correlation etc

teams are a whole different set of analysis, will need its own specialised tools

In [ ]:
cluster_teamcounts = []
for i in range(len(K_VALS)):
    k = K_VALS[i]
    cols = [f"col_{j}" for j in range(k)]
    target_df = k_clustered_dfs[i].copy()
    cur_df = pd.crosstab(columns = target_df[f"clusters_{k}"], index = [target_df["team_id"], target_df["match_id"]])
    cur_df.columns = cols
    cur_df["total_sum"] = cur_df[cols].sum(axis = 1)
    cluster_teamcounts.append(cur_df)

In [ ]:
team_k3_df = cluster_teamcounts[0]
team_k4_df = cluster_teamcounts[1]
team_k5_df = cluster_teamcounts[2]
team_k6_df = cluster_teamcounts[3]

In [ ]:
player_team_count = pm_df.groupby('player')['team_id'].nunique()